# Mini Project: Sales Data Cleaning with Pandas

**Dataset columns:** OrderID, CustomerName, City, Product, Quantity, UnitPrice, OrderDate, OrderStatus, CustomerAge

**Goal:** find and fix missing values, outliers, data-type errors, inconsistent text, extra spaces, duplicates and date problems, add a `TotalAmount` column, and export `cleaned_sales_dataset.csv`.

> **Before you run:** upload your CSV to Colab (folder icon on the left) and set `FILE_NAME` in the next cell to its exact name. If the file isn't found, the notebook will open an upload prompt instead.

## 1. Load the data

In [1]:
import re
import numpy as np
import pandas as pd

FILE_NAME = "Messy_data.csv"   # <-- change to your CSV's name

import os
if not os.path.exists(FILE_NAME):
    try:
        from google.colab import files
        uploaded = files.upload()
        FILE_NAME = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{FILE_NAME} not found")

# dtype=str keeps every value exactly as written, so bad entries like "three" or "18,500" aren't silently changed
raw = pd.read_csv(FILE_NAME, dtype=str, keep_default_na=True)
df = raw.copy()
raw.head(10)

,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
0,10001,Nikhil Jain,Mumbai,Laptop,3.0,50790.72,2026-01-18,Completed,61.0
1,10002,Simran Kaur,delhi,Monitor,1.0,10307.27,2026-01-28,Pending,50.0
2,10003,Zoya Ali,Delhi,Headphones,5.0,2439.64,2026-02-27,Delivered,35.0
3,10004,Priya Bansal,Delhi,Tablet,4.0,20945.65,02/03/2026,Pending,66.0
4,10005,NaN,Mumbai,Phone,4.0,15822.27,2026-02-14,Delivered,34.0
5,10006,Priya Bansal,Delhi,Smartwatch,5.0,5768.41,2026-02-18,Completed,53.0
6,10007,Meera Joshi,Chennai,Headphones,1.0,2159.37,2026-01-30,Shipped,23.0
7,10008,Sara Thomas,Bengaluru,Phone,three,16801.06,2026-03-23,Shipped,28.0
8,10009,Tara Kapoor,MUMBAI,Headphones,3.0,2651.37,2026-03-29,Completed,56.0
9,10010,Nikhil Jain,Pune,Headphones,2.0,2471.7,2026-02-04,Delivered,32.0


## 2. Inspect the dataset

In [2]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

Shape (rows, columns): (185, 9)

<class 'pandas.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   OrderID       185 non-null    str  
 1   CustomerName  183 non-null    str  
 2   City          183 non-null    str  
 3   Product       185 non-null    str  
 4   Quantity      183 non-null    str  
 5   UnitPrice     183 non-null    str  
 6   OrderDate     183 non-null    str  
 7   OrderStatus   183 non-null    str  
 8   CustomerAge   183 non-null    str  
dtypes: str(9)
memory usage: 13.1 KB


In [3]:
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


OrderID         0
CustomerName    2
City            2
Product         0
Quantity        2
UnitPrice       2
OrderDate       2
OrderStatus     2
CustomerAge     2
dtype: int64

In [4]:
print("Exact duplicate rows:", df.duplicated().sum())
print()
for col in ["City", "Product", "OrderStatus"]:
    print(f"--- {col} (raw unique values) ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

Exact duplicate rows: 5

--- City (raw unique values) ---
City
Bengaluru      33
Delhi          23
Mumbai         22
Pune           21
Kolkata        21
Noida          21
Chennai        20
Gurugram       18
NaN             2
 delhi          1
MUMBAI          1
pUnE            1
  bengaluru     1

--- Product (raw unique values) ---
Product
Phone         34
Smartwatch    33
Monitor       24
Mouse         22
Tablet        20
Laptop        18
Keyboard      18
Headphones    14
laptop         1
PHONE          1

--- OrderStatus (raw unique values) ---
OrderStatus
Delivered     42
Shipped       38
Completed     34
Pending       34
Cancelled     32
NaN            2
completed      1
SHIPPED        1
 pending       1



**What the inspection shows**

- Because everything was loaded as text, `info()` reports every column as `object`; the numeric and date columns need converting.
- `isnull().sum()` shows gaps in several columns (names, cities, quantities, prices, dates, ages).
- The raw `value_counts()` reveal the same city/status written in several ways (`delhi`, `DELHI`, `Delhi`) and values padded with spaces.
- Some rows are exact duplicates.

## 3. Clean text: extra spaces and consistent capitalisation

In [5]:
text_cols = ["CustomerName", "City", "Product", "OrderStatus"]

for col in text_cols:
    df[col] = (
        df[col]
        .str.strip()                          # leading / trailing spaces
        .str.replace(r"\s+", " ", regex=True)  # repeated inner spaces
        .str.title()                          # delhi / DELHI -> Delhi
    )

for col in ["City", "Product", "OrderStatus"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

--- City ---
City
Bengaluru    34
Delhi        24
Mumbai       23
Pune         22
Kolkata      21
Noida        21
Chennai      20
Gurugram     18
NaN           2

--- Product ---
Product
Phone         35
Smartwatch    33
Monitor       24
Mouse         22
Tablet        20
Laptop        19
Keyboard      18
Headphones    14

--- OrderStatus ---
OrderStatus
Delivered    42
Shipped      39
Completed    35
Pending      35
Cancelled    32
NaN           2



## 4. Fix data-type errors in numeric columns

Entries such as `"three"`, `"2 pcs"`, `"18,500"`, `"forty"` and `"unknown"` stop a column from being numeric. Rules used:

- number words → digits (`three` → 3, `forty` → 40)
- remove thousands separators and units (`18,500` → 18500, `2 pcs` → 2)
- anything that is not a number (`unknown`) → missing (`NaN`), to be handled in the missing-value step

In [6]:
WORDS = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
    "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
    "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
    "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90,
    "hundred": 100,
}

def to_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(",", "")
    if s in WORDS:
        return float(WORDS[s])
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group()) if m else np.nan

num_cols = ["Quantity", "UnitPrice", "CustomerAge"]
bad_before = {}
for col in num_cols:
    original = df[col].copy()
    df[col] = original.map(to_number)
    # entries that were present but are not plain numbers (e.g. "three", "18,500", "unknown")
    changed = original.notna() & pd.to_numeric(original, errors="coerce").isna()
    bad_before[col] = original[changed].unique().tolist()

df["OrderID"] = pd.to_numeric(df["OrderID"], errors="coerce")

print("Non-numeric / malformed entries found and converted:")
for col, vals in bad_before.items():
    print(f"  {col}: {vals}")
print()
df[num_cols].describe()

Non-numeric / malformed entries found and converted:
  Quantity: ['three', '2 pcs']
  UnitPrice: ['18,500', 'unknown']
  CustomerAge: ['forty', '29 years']



,Quantity,UnitPrice,CustomerAge
count,183.000000,182.000000,183.000000
mean,5.579235,19407.577527,44.819672
std,36.779608,74639.822854,18.544430
min,-3.000000,0.010000,3.000000
25%,2.000000,2459.100000,31.000000
50%,3.000000,10451.145000,45.000000
75%,4.000000,19373.162500,57.000000
max,500.000000,999999.000000,185.000000


## 5. Fix date inconsistencies

The dates use several formats and at least one is impossible (e.g. 30 Feb). Each value is tried against a list of known formats; day-first is assumed for slash/dash dates (Indian convention). Values that match none of them are treated as invalid and become missing (`NaT`).

In [7]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d", "%b %d, %Y", "%B %d, %Y",
                "%d %b %Y", "%d %B %Y", "%d.%m.%Y", "%m/%d/%Y"]

def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

date_text = df["OrderDate"].copy()
df["OrderDate"] = date_text.map(parse_date)

invalid_dates = date_text[date_text.notna() & df["OrderDate"].isna()]
print("Invalid / unparseable dates:", invalid_dates.tolist())
print("Date range:", df["OrderDate"].min().date(), "to", df["OrderDate"].max().date())

Invalid / unparseable dates: ['not a date']
Date range: 2026-01-01 to 2026-03-31


## 6. Remove duplicate records

Duplicates are removed **after** the text and date clean-up, because rows such as `" Delhi "` and `"Delhi"` only look identical once standardised.

In [8]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows ({before} -> {len(df)})")

# OrderID should be unique; check for any remaining repeats
print("Repeated OrderIDs remaining:", df["OrderID"].duplicated().sum())

Removed 5 duplicate rows (185 -> 180)
Repeated OrderIDs remaining: 0


## 7. Handle outliers

| Column | How outliers are detected | Treatment |
| --- | --- | --- |
| `CustomerAge` | Domain rule: valid ages are 18-100 | Out-of-range values → missing, then imputed |
| `UnitPrice` | Extreme IQR fence (Q3 + 3×IQR, Q1 − 3×IQR) **within each product**, plus prices ≤ 0 | Flagged → missing, then imputed |
| `Quantity` | Extreme IQR fence (3×IQR) and quantity ≤ 0 | Flagged → missing, then imputed |

Prices are compared within each product because a laptop is legitimately much more expensive than a mouse. The wider 3×IQR multiplier flags only *extreme* values, so genuine high-but-plausible orders are kept. Outliers are turned into missing values and handled together with the other gaps, rather than dropping whole rows, so no other information in those orders is lost.

In [9]:
def iqr_fences(s, k=3.0):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

log = {}

# Age: domain rule
age_bad = df["CustomerAge"].notna() & ~df["CustomerAge"].between(18, 100)
log["CustomerAge outliers"] = df.loc[age_bad, "CustomerAge"].tolist()
df.loc[age_bad, "CustomerAge"] = np.nan

# Price: per-product IQR fence
price_bad = pd.Series(False, index=df.index)
for prod, grp in df.groupby("Product"):
    s = grp["UnitPrice"].dropna()
    if len(s) < 4:
        continue
    lo, hi = iqr_fences(s)
    price_bad.loc[grp.index] = (grp["UnitPrice"] < lo) | (grp["UnitPrice"] > hi)
price_bad |= df["UnitPrice"] <= 0
log["UnitPrice outliers"] = df.loc[price_bad, "UnitPrice"].tolist()
df.loc[price_bad, "UnitPrice"] = np.nan

# Quantity: IQR fence + positive check
lo, hi = iqr_fences(df["Quantity"].dropna())
qty_bad = (df["Quantity"] > hi) | (df["Quantity"] <= 0) | (df["Quantity"] < lo)
log["Quantity outliers"] = df.loc[qty_bad, "Quantity"].tolist()
df.loc[qty_bad, "Quantity"] = np.nan

for k, v in log.items():
    print(f"{k}: {v}")

CustomerAge outliers: [185.0, 3.0]
UnitPrice outliers: [18500.0, 999999.0, 0.01, 709.84, 815.82]
Quantity outliers: [-3.0, 500.0]


## 8. Handle missing values

| Column | Strategy | Reason |
| --- | --- | --- |
| `CustomerName`, `City` | Fill with `"Unknown"` | Can't be inferred reliably; keeping the order is more useful than deleting it |
| `Quantity` | Median (whole number) | Median isn't distorted by the outliers just removed |
| `UnitPrice` | Median price of the same product | Price depends mainly on the product |
| `CustomerAge` | Median age | Simple, robust central value |
| `OrderDate` | Left as missing (`NaT`) | A date can't be guessed without inventing data; flagged so it can be reviewed |
| `OrderStatus`, `Product` | Fill with `"Unknown"` if any are missing | Keeps the column complete |

In [10]:
print("Missing values BEFORE imputation:")
print(df.isnull().sum().to_string())

for col in ["CustomerName", "City", "Product", "OrderStatus"]:
    df[col] = df[col].fillna("Unknown")

df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median()).round()

price_median_by_product = df.groupby("Product")["UnitPrice"].transform("median")
df["UnitPrice"] = df["UnitPrice"].fillna(price_median_by_product).fillna(df["UnitPrice"].median())

df["CustomerAge"] = df["CustomerAge"].fillna(df["CustomerAge"].median()).round()

print()
print("Missing values AFTER imputation:")
print(df.isnull().sum().to_string())

Missing values BEFORE imputation:
OrderID         0
CustomerName    2
City            2
Product         0
Quantity        4
UnitPrice       8
OrderDate       3
OrderStatus     2
CustomerAge     4

Missing values AFTER imputation:
OrderID         0
CustomerName    0
City            0
Product         0
Quantity        0
UnitPrice       0
OrderDate       3
OrderStatus     0
CustomerAge     0


## 9. Convert to suitable data types

In [11]:
df = df.astype({
    "OrderID": "int64",
    "CustomerName": "string",
    "City": "category",
    "Product": "category",
    "Quantity": "int64",
    "UnitPrice": "float64",
    "OrderStatus": "category",
    "CustomerAge": "int64",
})
# OrderDate is already datetime64[ns]
df.dtypes

OrderID                  int64
CustomerName            string
City                  category
Product               category
Quantity                 int64
UnitPrice              float64
OrderDate       datetime64[us]
OrderStatus           category
CustomerAge              int64
dtype: object

## 10. Create the `TotalAmount` column

In [12]:
df["TotalAmount"] = (df["Quantity"] * df["UnitPrice"]).round(2)
df[["OrderID", "Product", "Quantity", "UnitPrice", "TotalAmount"]].head()

,OrderID,Product,Quantity,UnitPrice,TotalAmount
0,10001,Laptop,3,50790.72,152372.16
1,10002,Monitor,1,10307.27,10307.27
2,10003,Headphones,5,2439.64,12198.20
3,10004,Tablet,4,20945.65,83782.60
4,10005,Phone,4,15822.27,63289.08


## 11. Final checks

In [13]:
print("Shape:", df.shape)
print()
df.info()
print()
print("Duplicates:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum(), "(only OrderDate may remain - see Section 8)")
print()
print("Cities:", sorted(df["City"].unique()))
print("Statuses:", sorted(df["OrderStatus"].unique()))
print()
df.describe()

Shape: (180, 10)

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   OrderID       180 non-null    int64         
 1   CustomerName  180 non-null    string        
 2   City          180 non-null    category      
 3   Product       180 non-null    category      
 4   Quantity      180 non-null    int64         
 5   UnitPrice     180 non-null    float64       
 6   OrderDate     177 non-null    datetime64[us]
 7   OrderStatus   180 non-null    category      
 8   CustomerAge   180 non-null    int64         
 9   TotalAmount   180 non-null    float64       
dtypes: category(3), datetime64[us](1), float64(2), int64(3), string(1)
memory usage: 10.7 KB



Duplicates: 0
Missing values: 3 (only OrderDate may remain - see Section 8)

Cities: ['Bengaluru', 'Chennai', 'Delhi', 'Gurugram', 'Kolkata', 'Mumbai', 'Noida', 'Pune', 'Unknown']
Statuses: ['Cancelled', 'Completed', 'Delivered', 'Pending', 'Shipped', 'Unknown']



,OrderID,Quantity,UnitPrice,OrderDate,CustomerAge,TotalAmount
count,180.000000,180.000000,180.000000,177,180.000000,180.000000
mean,10090.500000,2.905556,14785.384556,2026-02-18 18:18:18.305084,44.411111,43680.415222
min,10001.000000,1.000000,723.300000,2026-01-01 00:00:00,18.000000,723.300000
25%,10045.750000,2.000000,2498.280000,2026-01-28 00:00:00,31.000000,5764.170000
50%,10090.500000,3.000000,11061.240000,2026-02-21 00:00:00,45.000000,25523.160000
75%,10135.250000,4.000000,19521.737500,2026-03-12 00:00:00,57.000000,59623.275000
max,10180.000000,5.000000,62277.550000,2026-03-31 00:00:00,70.000000,302722.300000
std,52.105662,1.413017,15673.683191,NaN,14.998245,55753.622183


## 12. Export the cleaned dataset

In [14]:
df.to_csv("cleaned_sales_dataset.csv", index=False)
print("Saved cleaned_sales_dataset.csv with", len(df), "rows")

try:
    from google.colab import files
    files.download("cleaned_sales_dataset.csv")
except ImportError:
    pass

Saved cleaned_sales_dataset.csv with 180 rows


## 13. Summary of cleaning steps

1. **Loaded** the CSV as text so malformed values stayed visible, then inspected it with `head()`, `info()`, `shape` and `isnull().sum()`.
2. **Text:** trimmed leading/trailing and repeated spaces and converted names, cities, products and statuses to Title Case so `delhi`, `DELHI` and `Delhi` became one value.
3. **Data-type errors:** converted number words and strings like `2 pcs` and `18,500` to numbers; non-numeric entries such as `unknown` became missing.
4. **Dates:** parsed all formats into one `datetime` type; impossible dates became missing.
5. **Duplicates:** removed exact duplicate rows after standardising, so hidden duplicates were also caught.
6. **Outliers:** flagged unrealistic ages (outside 18-100), extreme prices (per product, 3×IQR) and extreme quantities (3×IQR); set them to missing rather than deleting the whole order.
7. **Missing values:** names/cities → `"Unknown"`; quantity and age → median; price → median of the same product; missing dates left as `NaT` rather than invented.
8. **Types and new column:** set suitable dtypes and added `TotalAmount = Quantity × UnitPrice`.
9. **Exported** `cleaned_sales_dataset.csv`.